<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week03_Search_BFS_DFS_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 3 Practical (PH):** Problem solving by search — uninformed strategies and combinatorial explosion

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO5

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — Search strategies from first principles | Task A1 — Implement **uniform-cost search** |
| Part B — Missionaries & cannibals (CCMAS lab problem) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part C — Combinatorial explosion | Task C0 — Trace the frontier after every iteration<br>Task C1 — Effective branching factor $b^{*}$ |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | formulate a problem as a **state space** $(S, s_0, \text{Actions}, T, G, c)$ | Apply | LO5 |
| 2 | implement **BFS, DFS, depth-limited, iterative-deepening and uniform-cost** search | Apply | LO5 |
| 3 | compare strategies on **completeness, optimality, time and space** | Analyse | LO5 |
| 4 | measure **combinatorial explosion** and estimate the effective branching factor $b^{*}$ | Analyse / Evaluate | LO5 |

**Formal definition.** A search problem is a tuple $(S, s_0, A, T, G, c)$: states $S$, initial state $s_0$, actions $A(s)$, transition $T(s,a)=s'$, goal set $G\subseteq S$ and step cost $c(s,a,s')>0$. A *solution* is an action sequence from $s_0$ to some $g\in G$; it is *optimal* if its path cost $\sum c$ is minimal.

**Complexity notation.** $b$ = branching factor, $d$ = depth of the shallowest goal, $m$ = maximum depth, $\ell$ = depth limit, $C^{*}$ = optimal cost, $\varepsilon$ = minimum step cost.

| Strategy | Complete? | Optimal? | Time | Space |
|---|---|---|---|---|
| BFS | yes (finite $b$) | yes if unit costs | $O(b^{d})$ | $O(b^{d})$ |
| DFS (tree search) | no (loops / infinite depth) | no | $O(b^{m})$ | $O(bm)$ |
| Depth-limited | no (if $\ell<d$) | no | $O(b^{\ell})$ | $O(b\ell)$ |
| Iterative deepening | yes | yes if unit costs | $O(b^{d})$ | $O(bd)$ |
| Uniform-cost | yes if $\varepsilon>0$ | **yes** | $O(b^{1+\lfloor C^{*}/\varepsilon\rfloor})$ | $O(b^{1+\lfloor C^{*}/\varepsilon\rfloor})$ |

**Do Now (3 min):** a maze has branching factor 3 and the exit is 12 steps away. Roughly how many nodes might BFS generate? Write your guess, then check it in Part C.

In [ ]:
#@title 🗺️ Problem interface + Romania road map (Russell & Norvig) — shared by Weeks 3–4
# heapq = priority queue (used by UCS/A*); itertools gives us counters for fresh names
import heapq, itertools
# deque = double-ended queue; popleft() makes a fast first-in-first-out queue (BFS)
from collections import deque
# Positive infinity: a cost that is larger than any real cost
INF = float("inf")

class Problem:
    """A search problem = initial state + successor function + goal test (+ step costs)."""
    # Every search problem must say where we start; subclasses override this
    def initial(self):            raise NotImplementedError
    # ...and how to test whether a state is the goal
    def is_goal(self, s):         raise NotImplementedError
    # ...and which (action, next_state, step_cost) moves are legal from a state
    def successors(self, s):      raise NotImplementedError      # → list of (action, next_state, step_cost)

# A problem defined by a weighted graph (e.g. a road map)
class GraphProblem(Problem):
    # Constructor: edges is a list of (city_a, city_b, distance)
    def __init__(self, edges, start, goal):
        # Adjacency dictionary: city -> list of (action, neighbour, cost)
        self.adj = {}
        # Unpack each road into its two ends a, b and its length d
        for a, b, d in edges:
            # Add the move a -> b (setdefault creates the list the first time we see a)
            self.adj.setdefault(a, []).append((f"go {b}", b, d))
            # Roads are two-way, so also add the move b -> a
            self.adj.setdefault(b, []).append((f"go {a}", a, d))
        # Remember the start and goal cities (tuple assignment: two names in one line)
        self.start, self.goal = start, goal
    # The search starts in the start city
    def initial(self):       return self.start
    # A state is the goal if it equals the goal city
    def is_goal(self, s):    return s == self.goal
    # Legal moves from s; .get returns [] for a city with no roads
    def successors(self, s): return self.adj.get(s, [])

# The Romania road map from AIMA: (city, city, kilometres) triples
ROMANIA_EDGES = [("Arad","Zerind",75),("Arad","Sibiu",140),("Arad","Timisoara",118),("Zerind","Oradea",71),
    ("Oradea","Sibiu",151),("Sibiu","Fagaras",99),("Sibiu","RimnicuVilcea",80),("Timisoara","Lugoj",111),
    ("Lugoj","Mehadia",70),("Mehadia","Drobeta",75),("Drobeta","Craiova",120),("Craiova","RimnicuVilcea",146),
    ("Craiova","Pitesti",138),("RimnicuVilcea","Pitesti",97),("Pitesti","Bucharest",101),("Fagaras","Bucharest",211)]
# Straight-line distance to Bucharest, used as the A* heuristic h(n)
SLD_TO_BUCHAREST = {"Arad":366,"Bucharest":0,"Craiova":160,"Drobeta":242,"Fagaras":176,"Lugoj":244,"Mehadia":241,
    "Oradea":380,"Pitesti":100,"RimnicuVilcea":193,"Sibiu":253,"Timisoara":329,"Zerind":374}
# Build the Romania problem: start Arad, goal Bucharest
romania = GraphProblem(ROMANIA_EDGES, "Arad", "Bucharest")

# The 8-puzzle: a 3x3 board with tiles 1-8 and one blank
class EightPuzzle(Problem):
    # Goal board read row by row; 0 stands for the blank
    GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)                       # 0 = blank
    # Store the start board as a tuple (tuples are hashable, so they can go in sets)
    def __init__(self, start): self.start = tuple(start)
    # The search starts from the given board
    def initial(self):    return self.start
    # Goal test: is the board exactly the goal arrangement?
    def is_goal(self, s): return s == self.GOAL
    # List every legal slide of the blank
    def successors(self, s):
        # i = blank position; divmod gives (row, column) on the 3x3 grid; out collects moves
        i = s.index(0); r, c = divmod(i, 3); out = []
        # Try each direction: name and the row/column change (dr, dc)
        for name, dr, dc in (("Up", -1, 0), ("Down", 1, 0), ("Left", 0, -1), ("Right", 0, 1)):
            # Where the blank would land after the move
            nr, nc = r + dr, c + dc
            # Only allow the move if it stays on the board
            if 0 <= nr < 3 and 0 <= nc < 3:
                # j = index of that cell; swap blank and neighbour on a list copy
                j = nr * 3 + nc; l = list(s); l[i], l[j] = l[j], l[i]
                # Record (action name, new board as tuple, cost 1)
                out.append((name, tuple(l), 1))
        # Hand back all legal moves
        return out

def scramble(k, rng):
    """Random walk of k moves away from the goal (never undoing the previous move) → always solvable."""
    # Start at the goal and scramble it (guarantees the puzzle is solvable)
    s = EightPuzzle.GOAL; prev = None; p = EightPuzzle(s)
    # Repeat the random move k times
    for _ in range(k):
        # All successor boards except the one that undoes the previous move
        opts = [t for (_, t, _) in p.successors(s) if t != prev]
        # Remember where we were; pick a random successor using the seeded generator
        prev, s = s, opts[rng.integers(0, len(opts))]
    # Return the scrambled board
    return s

## Part A — Search strategies from first principles
Every strategy is the **same loop** with a different *frontier* data structure:

| Strategy | Frontier | Pops… |
|---|---|---|
| BFS | FIFO queue | oldest node |
| DFS | LIFO stack | newest node |
| UCS | priority queue on $g(n)$ | cheapest path so far |

We count three things: **expanded** (time), **generated** and **max frontier size** (space).

### 🔍 Code walkthrough — one loop, two behaviours
`graph_search` is the same loop for BFS and DFS. The *only* difference is which end of the frontier we take from:
- `popleft()` takes the **oldest** state, so the frontier is a **queue** → breadth-first.
- `pop()` takes the **newest** state, so the frontier is a **stack** → depth-first.

The dictionary `parent` has two jobs: it stops us visiting a state twice, and it lets us rebuild the path at the end by walking back from the goal. Trace the loop by hand on a 3-node graph before running it.

In [ ]:
#@title 🧭 BFS, DFS (graph search), depth-limited and iterative-deepening search
# Helper that rebuilds the solution path from parent links
def _result(parent, g, s, expanded, generated, max_frontier):
    # Start with an empty path
    path = []
    # Begin at the goal node
    node = s
    # Walk back to the start (the start has parent None)
    while node is not None:
        # Add the node, then move to its parent
        path.append(node); node = parent[node]
    # Reverse the path and return it with the cost and statistics
    return {"path": path[::-1], "cost": g[s], "expanded": expanded, "generated": generated, "max_frontier": max_frontier}

def graph_search(problem, kind="bfs"):
    """BFS (FIFO) or DFS (LIFO) with an explored set; the goal test is applied when a node is expanded."""
    # Start state of the problem
    s0 = problem.initial()
    # Frontier (queue/stack), parent links, and path cost g for each state
    frontier = deque([s0]); parent = {s0: None}; g = {s0: 0}
    # Counters for the statistics
    expanded, generated, max_frontier = 0, 1, 1
    # Loop while there are states left to explore
    while frontier:
        # BFS takes from the front (queue); DFS takes from the back (stack)
        s = frontier.popleft() if kind == "bfs" else frontier.pop()
        # Count this expansion
        expanded += 1
        # Goal test when a state is taken off the frontier
        if problem.is_goal(s):
            # Return the solution and statistics
            return _result(parent, g, s, expanded, generated, max_frontier)
        # Look at each successor (action, state, step cost)
        for _, n, c in problem.successors(s):
            # Only add states we have not seen yet
            if n not in parent:                              # never revisit a state
                # Record parent and cost; push to the frontier
                parent[n] = s; g[n] = g[s] + c; frontier.append(n); generated += 1
        # Track the largest frontier (memory used)
        max_frontier = max(max_frontier, len(frontier))
    # Frontier empty: no solution
    return None

# Breadth-first search = graph_search with a queue
bfs = lambda p: graph_search(p, "bfs")
# Depth-first search = graph_search with a stack
dfs = lambda p: graph_search(p, "dfs")

def dls(problem, limit):
    """Recursive depth-limited search. Returns (path | None | 'cutoff', expanded, generated)."""
    # Counters shared with the inner function
    stats = {"expanded": 0, "generated": 1}
    # Recursive depth-limited search
    def rec(s, depth, on_path):
        # Count each expansion
        stats["expanded"] += 1
        # Goal found: return the path
        if problem.is_goal(s): return [s]
        # Depth limit reached: report a cutoff, not failure
        if depth == limit:     return "cutoff"
        # Track whether any branch was cut off
        cutoff = False
        # Try each successor
        for _, n, _ in problem.successors(s):
            # Skip states already on the current path (avoid loops)
            if n in on_path: continue                        # avoid cycles along the current path only
            # Count generated nodes
            stats["generated"] += 1
            # Recurse one level deeper
            r = rec(n, depth + 1, on_path | {n})
            # Remember that a cutoff occurred
            if r == "cutoff": cutoff = True
            # A real solution: prepend this state and return it
            elif r is not None: return [s] + r
        # Either some branch was cut off, or the search failed
        return "cutoff" if cutoff else None
    # Start the recursion at the start state
    r = rec(problem.initial(), 0, {problem.initial()})
    # Return the result and the statistics
    return r, stats["expanded"], stats["generated"]

def ids(problem, max_depth=60):
    """Iterative deepening: DLS with limit 0,1,2,… (BFS's optimality with DFS's memory)."""
    # Totals over all iterations
    tot_e = tot_g = 0
    # Try depth limits 0, 1, 2, ...
    for L in range(max_depth + 1):
        # Run depth-limited search with limit L and add its counts
        r, e, g_ = dls(problem, L); tot_e += e; tot_g += g_
        # Stop at the first limit that does not cut off
        if r != "cutoff":
            # Return the solution and statistics (max_frontier = depth reached)
            return {"path": r, "cost": (len(r) - 1) if r else None, "expanded": tot_e, "generated": tot_g, "max_frontier": L}
    # Frontier empty: no solution
    return None

### Task A1 — Implement **uniform-cost search**
UCS expands the node with the **lowest path cost $g(n)$** using a priority queue (`heapq`). It must be **optimal** for any positive step costs. The Romania road map below has real distances, so the route with the *fewest* cities is **not** the cheapest. Return the same dictionary format as `bfs`.

In [ ]:
# TODO(Student): implement ucs(problem) with heapq (priority = path cost g)
# TODO: uniform-cost search (expand the cheapest path first)
def ucs(problem):
    # Start state and a tie-breaker counter for the priority queue
    s0 = problem.initial(); tie = itertools.count()          # tie-breaker so heapq never compares states
    # Priority queue of (cost, tie, state)
    frontier = [(0, next(tie), s0)]
    # …
    # Frontier empty: no solution
    return None

In [ ]:
#@title ✅ Auto-check for Task A1 + comparison on the Romania map
# Run BFS and UCS on the Romania map
r_bfs, r_ucs = bfs(romania), ucs(romania)
# Show BFS: fewest steps
print("BFS:", r_bfs["path"], "cost", r_bfs["cost"])
# Show UCS: cheapest distance
print("UCS:", r_ucs["path"] if r_ucs else None, "cost", r_ucs["cost"] if r_ucs else None)
check("UCS finds the cheapest route (cost 418)",  lambda: r_ucs["cost"] == 418)
check("UCS route: Arad→Sibiu→RimnicuVilcea→Pitesti→Bucharest", lambda: r_ucs["path"] == ["Arad","Sibiu","RimnicuVilcea","Pitesti","Bucharest"])
check("BFS finds fewest hops but NOT the cheapest (cost 450)", lambda: r_bfs["cost"] == 450 and len(r_bfs["path"]) == 4)

## Part B — Missionaries & cannibals (CCMAS lab problem)
Three missionaries and three cannibals must cross a river in a boat that carries 1 or 2 people. On **either** bank (and never on the boat's account) cannibals may not outnumber missionaries unless there are no missionaries.

**State** $(M_L, C_L, \beta)$: missionaries and cannibals on the *left* bank, and $\beta=1$ if the boat is on the left. Start $(3,3,1)$; goal $(0,0,0)$.

In [ ]:
#@title ⛵ Missionaries & cannibals as a Problem
# Missionaries and cannibals puzzle as a search problem
class MissionariesCannibals(Problem):
    # Boat loads allowed: (missionaries, cannibals)
    LOADS = [(1, 0), (2, 0), (0, 1), (0, 2), (1, 1)]
    # State = (missionaries on left, cannibals on left, boat on left?)
    def initial(self):    return (3, 3, 1)
    # Everyone safely on the right bank
    def is_goal(self, s): return s == (0, 0, 0)
    # Utility method that does not need an object
    @staticmethod
    # A bank is safe if there are no missionaries or they are not outnumbered
    def safe(M, C):       # bank is safe if no missionaries, or missionaries ≥ cannibals
        # Safety test
        return M == 0 or M >= C
    # Successor function
    def successors(self, s):
        # Unpack the state
        M, C, b = s; out = []
        # Try every boat load
        for m, c in self.LOADS:
            # Move the load from the boat's current side to the other
            M2, C2 = (M - m, C - c) if b == 1 else (M + m, C + c)
            # Check counts are valid and both banks are safe
            if 0 <= M2 <= 3 and 0 <= C2 <= 3 and self.safe(M2, C2) and self.safe(3 - M2, 3 - C2):
                # Record the move, new state and cost 1
                out.append((f"{m}M{c}C {'→' if b == 1 else '←'}", (M2, C2, 1 - b), 1))
        # All legal moves
        return out

# Create the problem
mc = MissionariesCannibals()
# Rows of results
rows = []
# Compare four strategies
for name, fn in [("BFS", bfs), ("DFS", dfs), ("UCS", ucs), ("IDS", ids)]:
    # Run the strategy
    r = fn(mc)
    # Placeholder row if the strategy is not finished yet
    if r is None:                                             # e.g. your ucs() is not finished yet
        # Skip to the next strategy
        rows.append((name, -1, -1, -1, -1)); continue
    # Store cost, nodes expanded, generated, memory
    rows.append((name, r["cost"], r["expanded"], r["generated"], r["max_frontier"]))
# Print the table header
print(f"{'strategy':<6}{'cost':>6}{'expanded':>10}{'generated':>11}{'max frontier / depth':>22}")
# Print one line per strategy
for row in rows: print(f"{row[0]:<6}{row[1]:>6}{row[2]:>10}{row[3]:>11}{row[4]:>22}")
check("BFS solves it in 11 crossings (the optimum)", lambda: rows[0][1] == 11)
check("IDS is also optimal here (11)",               lambda: rows[3][1] == 11)
check("DFS finds *a* solution (not necessarily shortest)", lambda: rows[1][1] >= 11)

**Discuss.** Which strategy used the *least memory*? Which gave the *shortest* solution? Why did DFS (graph search with an explored set) behave well on this tiny state space but would not in an infinite one?

## Part C — Combinatorial explosion
The number of nodes a blind search must visit grows like $b^{d}$. Doubling the depth **squares** the work. Run the cell, then compare with your Do Now guess.

In [ ]:
#@title 💥 How fast does b^d grow?  (and how big are real state spaces?)
# Explain the exponential growth table
print("Nodes at depth d for branching factor b (and time at 10⁹ nodes/second):")
# Table header
print(f"{'b':>3} {'d':>3} {'b^d':>12}   time")
# Try several branching factors and depths
for b, d in [(3, 12), (10, 8), (10, 12), (10, 16), (35, 20)]:
    # Nodes n = b^d and seconds at 10^9 nodes per second
    n = b ** d; sec = n / 1e9
    # Convert seconds to a readable unit
    t = (f"{sec:.2g} s" if sec < 60 else f"{sec/3600:.2g} h" if sec < 86400 else f"{sec/86400:.2g} days" if sec < 3.15e7 else f"{sec/3.15e7:.2g} years")
    # Print one row
    print(f"{b:>3} {d:>3} {n:>12.3g}   {t}")

# Real state-space sizes
print("\nSize of real state spaces:")
# 8-puzzle: 9!/2 reachable states
print("8-puzzle    reachable states 9!/2   =", f"{math.factorial(9)//2:,}")
# 15-puzzle: 16!/2
print("15-puzzle   reachable states 16!/2  =", f"{math.factorial(16)//2:,}")
# TSP tours: (n-1)!/2
print("TSP, 20 cities  distinct tours (n-1)!/2 =", f"{math.factorial(19)//2:.3e}")
# Chess game tree estimate (Shannon)
print("Chess game tree ≈ 10^120 (Shannon's 1950 estimate); chess positions ≈ 10^44–10^47")

### Task C0 — Trace the frontier after every iteration (class note, Examples 3.2–3.3)
Before trusting a search implementation, *watch* it. On the 7-node binary tree of the class note (A is the root; B, C its children; D, E under B; F, G under C), print the frontier after each expansion for BFS (FIFO) and DFS (LIFO). Then check the cost of iterative deepening against BFS: with branching factor $b$ and solution depth $d$,
$$N_{\mathrm{BFS}}=\sum_{i=0}^{d}b^{i},\qquad N_{\mathrm{IDS}}=\sum_{i=0}^{d}(d+1-i)\,b^{i}.$$

In [ ]:
# TODO(Student): trace(kind) → list of (expanded_node, frontier_after) and the IDS/BFS node counts
# A small tree given as a dictionary of children
TREE = {"A": ["B", "C"], "B": ["D", "E"], "C": ["F", "G"], "D": [], "E": [], "F": [], "G": []}

def trace(kind="bfs"):
    """Expand nodes of TREE from 'A'. BFS pops the oldest node, DFS the newest (push children so B is popped before C).
    Return the list of (node_expanded, list(frontier_after_expansion))."""
    # Placeholder return; fill in your own answer
    return []

# TODO: nodes generated by BFS
def n_bfs(b, d): return None
# TODO: nodes generated by iterative deepening
def n_ids(b, d): return None

In [ ]:
#@title ✅ Auto-check and trace table for Task C0
# Trace both strategies
for kind in ("bfs", "dfs"):
    # Print a heading
    print(f"\n{kind.upper()}:  expanded → frontier after expansion")
    # Print each expansion step
    for node, fr in trace(kind): print(f"   {node} → {fr}")
check("BFS visits A,B,C,D,E,F,G",  lambda: [n for n, _ in trace("bfs")] == list("ABCDEFG"))
check("DFS visits A,B,D,E,C,F,G",  lambda: [n for n, _ in trace("dfs")] == list("ABDECFG"))
check("N_BFS(10,5) = 111,111",     lambda: n_bfs(10, 5) == 111_111)
check("N_IDS(10,5) = 123,456 (only 11% more than BFS)", lambda: n_ids(10, 5) == 123_456 and abs(n_ids(10, 5) / n_bfs(10, 5) - 1.111) < 0.001)
check("Exercise 3.1: b=3, d=4 → 121 and 179",           lambda: n_bfs(3, 4) == 121 and n_ids(3, 4) == 179)
# Years needed to enumerate all TSP tours at 10^9 per second
t = math.factorial(19) / 2 / 1e9 / 3.15e7
check("Example 3.4: 20-city TSP at 10⁹ tours/s ≈ 1.9 years", lambda: abs(t - 1.93) < 0.02)

### Task C1 — Effective branching factor $b^{*}$
If a search generates $N$ nodes and the solution is at depth $d$, the **effective branching factor** $b^{*}$ is the branching factor of a *uniform* tree of depth $d$ that would contain $N+1$ nodes:
$$N + 1 = 1 + b^{*} + (b^{*})^{2} + \dots + (b^{*})^{d}.$$
Solve for $b^{*}$ numerically (bisection on $b^{*}\in[1, N]$). *A small $b^{*}$ means a good search; $b^{*}=1$ is perfect.*

In [ ]:
# TODO(Student): effective_branching_factor(N, d) by bisection
# Effective branching factor: the b* with 1 + b* + ... + b*^d = N + 1
def effective_branching_factor(N, d, tol=1e-6):
    # Placeholder; replace with your solver
    return float("nan")

In [ ]:
#@title ✅ Auto-check for Task C1
check("N=52, d=5 → b* ≈ 1.92 (Russell & Norvig's example)", lambda: abs(effective_branching_factor(52, 5) - 1.92) < 0.01)
check("a perfect chain (N=d) → b* ≈ 1",                      lambda: abs(effective_branching_factor(5, 5) - 1.0) < 0.01)

### Experiment — BFS on the 8-puzzle
Scramble the goal by $k$ random moves (so every instance is solvable), solve with BFS, and record the **solution depth** $d$ and **nodes generated** $N$. Use 15 instances per scramble length and report $b^{*}$ with a 95% CI.

In [ ]:
#@title 🧪 Measure the explosion (8-puzzle, BFS)
# Plotting library
import matplotlib.pyplot as plt
# Store (depth, nodes generated, b*) per run
data = []                                                    # (d, N, b*)
# Scramble depths 4 to 20
for k in [4, 8, 12, 16, 20]:
    # 15 puzzles per depth
    for i in range(15):
        # Seeded generator for reproducibility
        rng = np.random.default_rng(SEED + 1000 * k + i)
        # Solve with BFS and read off the depth
        r = bfs(EightPuzzle(scramble(k, rng))); d = len(r["path"]) - 1
        # Save the results
        data.append((d, r["generated"], effective_branching_factor(r["generated"], d)))
# Convert to an array
data = np.array(data)
# Report b* statistics
report("b* (BFS, 8-puzzle)", data[:, 2])

# Create figure
plt.figure(figsize=(6, 3.6))
# Scatter: nodes generated vs depth, log scale
plt.semilogy(data[:, 0], data[:, 1], "o", alpha=0.6, label="BFS nodes generated")
# Depth values for the curve
dd = np.arange(1, int(data[:, 0].max()) + 1)
# Theoretical curve b*^d
plt.semilogy(dd, [(data[:, 2].mean()) ** x for x in dd], "r--", label=f"$b^d$ with $b^*$ = {data[:, 2].mean():.2f}")
# Reference line: the entire state space
plt.axhline(math.factorial(9) // 2, color="gray", ls=":", label="all reachable states = 181,440")
# Labels, legend, display
plt.xlabel("solution depth d"); plt.ylabel("nodes generated (log scale)"); plt.legend(); plt.tight_layout(); plt.show()
check("effective branching factor is between 1 and 3", lambda: 1.0 < data[:, 2].mean() < 3.0)
check("deeper solutions cost more nodes (rank corr > 0.8)", lambda: __import__("scipy.stats", fromlist=["spearmanr"]).spearmanr(data[:, 0], data[:, 1])[0] > 0.8)

**Consequences of combinatorial explosion** (write 3 bullet points in your reflection): (i) blind search is only feasible for *small* depth; (ii) memory runs out before time does for BFS; (iii) we need **heuristics** (Week 4) to prune the space, accepting *incompleteness* or *sub-optimality* as the price.

## 📝 Assignment (5 marks, individual, formative)
1. *(3)* Model a **new puzzle** as a `Problem` subclass (the 3-litre/5-litre water-jug problem is a good choice). Compare BFS, DFS, UCS and IDS in one table: cost, nodes expanded, nodes generated, memory.
2. *(1)* Estimate the effective branching factor $b^*$ on 10 instances: mean ± sd and 95% CI.
3. *(1)* One paragraph: for which kind of problem would you choose each of the four strategies, and why?

*Marked with the course rubric (class note, Table 0.3): correctness, evidence (numbers with CI and a named test), code clarity, interpretation. Submit the **executed notebook** (Runtime → Run all, outputs visible) as a Colab share link or `.ipynb` before the next lab.*

## Reflection
- For which problems would you choose DFS, BFS, UCS or IDS? Give one example each.
- Why does IDS repeat work yet still cost $O(b^{d})$?

```text
Write your reflection here…
```

## Extensions
- Solve the **water-jug** problem (4 L and 3 L jugs, measure exactly 2 L) by defining a new `Problem` subclass.
- Add *bidirectional search* and compare its node count with BFS on the 8-puzzle.
- Show that DFS *tree* search (no explored set) loops on a graph with a cycle by running it with a depth cap.

In [ ]:
checks_summary()  # how many auto-checks have you passed?